In [1]:
import pandas as pd
from pathlib import Path

In [2]:
processed_path = Path("../data/processed")

X_train = pd.read_csv(processed_path / "X_train.csv")
y_train = pd.read_csv(processed_path / "y_train.csv").squeeze()

X_test = pd.read_csv(processed_path / "X_test.csv")
y_test = pd.read_csv(processed_path / "y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (1952893, 70)
y_train: (1952893,)
X_test: (621371, 70)
y_test: (621371,)


In [3]:
print("Training target:")
print(y_train.value_counts())

print("\nTesting target:")
print(y_test.value_counts())

Training target:
Target
0    1747803
1     205090
Name: count, dtype: int64

Testing target:
Target
0    400583
1    220788
Name: count, dtype: int64


In [4]:
%pip install -q scikit-learn

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (1952893, 70)
X_test_scaled shape: (621371, 70)


In [6]:
model_lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    n_jobs=-1
)

model_lr.fit(X_train_scaled, y_train)

print("Logistic Regression training completed.")

d:\secureAccessControl\.venv-1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression training completed.


In [7]:
y_pred_lr = model_lr.predict(X_test_scaled)

print("Predictions completed.")

Predictions completed.


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_lr)
precision = precision_score(y_test, y_pred_lr)
recall = recall_score(y_test, y_pred_lr)
f1 = f1_score(y_test, y_pred_lr)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.8347863031908473
Precision: 0.912879661668589
Recall   : 0.591481421091726
F1 Score : 0.7178480828269336


In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_lr)

print(cm)

[[388120  12463]
 [ 90196 130592]]


In [10]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.81      0.97      0.88    400583
           1       0.91      0.59      0.72    220788

    accuracy                           0.83    621371
   macro avg       0.86      0.78      0.80    621371
weighted avg       0.85      0.83      0.82    621371



In [11]:
baseline_results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

baseline_results

{'Model': 'Logistic Regression',
 'Accuracy': 0.8347863031908473,
 'Precision': 0.912879661668589,
 'Recall': 0.591481421091726,
 'F1 Score': 0.7178480828269336}

In [12]:
from sklearn.ensemble import RandomForestClassifier

In [13]:
model_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created.")

Random Forest model created.


In [14]:
model_rf.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [15]:
y_pred_rf = model_rf.predict(X_test)

print("Random Forest predictions completed.")

Random Forest predictions completed.


In [16]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)

Accuracy : 0.6474409008466762
Precision: 0.9441571871768356
Recall   : 0.008270377013243473
F1 Score : 0.016397122870663878


In [17]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

print(cm_rf)

[[400475    108]
 [218962   1826]]


In [18]:
print(pd.Series(y_pred_rf).value_counts())

0    619437
1      1934
Name: count, dtype: int64


In [19]:
print("Training attack percentage:",
      y_train.mean() * 100)

print("Testing attack percentage:",
      y_test.mean() * 100)

Training attack percentage: 10.501855452398058
Testing attack percentage: 35.53239530007033


In [20]:
from sklearn.model_selection import train_test_split

X_train_model, X_val, y_train_model, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.1,
    stratify=y_train,
    random_state=42
)

print("Training set:", X_train_model.shape)
print("Validation set:", X_val.shape)

Training set: (1757603, 70)
Validation set: (195290, 70)


In [21]:
print("Training set:")
print(y_train_model.value_counts())

print("\nValidation set:")
print(y_val.value_counts())

Training set:
Target
0    1573022
1     184581
Name: count, dtype: int64

Validation set:
Target
0    174781
1     20509
Name: count, dtype: int64


In [22]:
model_lr_val = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    n_jobs=-1
)

model_lr_val.fit(X_train_model, y_train_model)

print("Logistic Regression training completed.")

d:\secureAccessControl\.venv-1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\secureAccessControl\.venv-1\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression training completed.


In [23]:
scaler_val = StandardScaler()

X_train_model_scaled = scaler_val.fit_transform(X_train_model)
X_val_scaled = scaler_val.transform(X_val)

print("Training data scaled:", X_train_model_scaled.shape)
print("Validation data scaled:", X_val_scaled.shape)

Training data scaled: (1757603, 70)
Validation data scaled: (195290, 70)


In [24]:
model_lr_val = LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)

model_lr_val.fit(X_train_model_scaled, y_train_model)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [25]:
y_pred_lr_val = model_lr_val.predict(X_val_scaled)

print("Validation predictions completed.")

Validation predictions completed.


In [26]:
accuracy_val = accuracy_score(y_val, y_pred_lr_val)
precision_val = precision_score(y_val, y_pred_lr_val)
recall_val = recall_score(y_val, y_pred_lr_val)
f1_val = f1_score(y_val, y_pred_lr_val)

print("Validation Accuracy :", accuracy_val)
print("Validation Precision:", precision_val)
print("Validation Recall   :", recall_val)
print("Validation F1 Score :", f1_val)

Validation Accuracy : 0.9700087050028163
Validation Precision: 0.7861495195687837
Validation Recall   : 0.9813740309132576
Validation F1 Score : 0.8729804168202815


In [27]:
cm_val = confusion_matrix(y_val, y_pred_lr_val)

print(cm_val)

[[169306   5475]
 [   382  20127]]


In [28]:
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Validation": [
        accuracy_val,
        precision_val,
        recall_val,
        f1_val
    ],
    "Friday Test": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

print(comparison)

      Metric  Validation  Friday Test
0   Accuracy    0.970009     0.834786
1  Precision    0.786150     0.912880
2     Recall    0.981374     0.591481
3   F1 Score    0.872980     0.717848


In [29]:
y_prob_val = model_lr_val.predict_proba(X_val_scaled)[:, 1]

print(y_prob_val[:10])

[2.26536872e-006 9.99993092e-001 7.30596145e-005 1.03912380e-004
 2.39885990e-003 9.89656301e-149 1.40659829e-168 1.56818354e-009
 1.76354805e-165 9.99994548e-001]


In [30]:
for threshold in [0.3, 0.5, 0.7]:
    y_pred_threshold = (y_prob_val >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_val, y_pred_threshold))
    print("Recall   :", recall_score(y_val, y_pred_threshold))
    print("F1 Score :", f1_score(y_val, y_pred_threshold))


Threshold: 0.3
Precision: 0.6834717941846442
Recall   : 0.9948315373738359
F1 Score : 0.8102698516709358

Threshold: 0.5
Precision: 0.7861495195687837
Recall   : 0.9813740309132576
F1 Score : 0.8729804168202815

Threshold: 0.7
Precision: 0.9665403083143795
Recall   : 0.9324199132088351
F1 Score : 0.9491735742294138


In [31]:
X_test_scaled_val = scaler_val.transform(X_test)

print("Test data scaled:", X_test_scaled_val.shape)

Test data scaled: (621371, 70)


In [32]:
y_prob_test = model_lr_val.predict_proba(X_test_scaled_val)[:, 1]

print("Test probabilities calculated.")
print(y_prob_test[:10])

Test probabilities calculated.
[3.80566008e-153 1.10444096e-154 1.07071956e-154 3.76742486e-130
 3.85481616e-153 5.59948023e-153 2.88132491e-153 8.63186240e-154
 3.65677063e-026 1.83267400e-153]


In [33]:
threshold = 0.7

y_pred_test = (y_prob_test >= threshold).astype(int)

print("Predictions generated.")
print("Predicted BENIGN:", (y_pred_test == 0).sum())
print("Predicted ATTACK:", (y_pred_test == 1).sum())

Predictions generated.
Predicted BENIGN: 510615
Predicted ATTACK: 110756


In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

accuracy_test_07 = accuracy_score(y_test, y_pred_test)
precision_test_07 = precision_score(y_test, y_pred_test)
recall_test_07 = recall_score(y_test, y_pred_test)
f1_test_07 = f1_score(y_test, y_pred_test)

print("Accuracy :", accuracy_test_07)
print("Precision:", precision_test_07)
print("Recall   :", recall_test_07)
print("F1 Score :", f1_test_07)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_test))

Accuracy : 0.8185206583506472
Precision: 0.9876575535411174
Recall   : 0.4954481221805533
F1 Score : 0.6598762155249379

Confusion Matrix:
[[399216   1367]
 [111399 109389]]


In [35]:
results = pd.DataFrame({
    "Dataset": ["Validation", "Friday Test"],
    "Threshold": [0.7, 0.7],
    "Accuracy": [accuracy_val, accuracy_test_07],
    "Precision": [precision_score(y_val, (y_prob_val >= 0.7).astype(int)), precision_test_07],
    "Recall": [recall_score(y_val, (y_prob_val >= 0.7).astype(int)), recall_test_07],
    "F1 Score": [f1_score(y_val, (y_prob_val >= 0.7).astype(int)), f1_test_07]
})

results

,Dataset,Threshold,Accuracy,Precision,Recall,F1 Score
0,Validation,0.7,0.970009,0.966540,0.932420,0.949174
1,Friday Test,0.7,0.818521,0.987658,0.495448,0.659876


In [36]:
%pip install xgboost

import xgboost as xgb

print("XGBoost version:", xgb.__version__)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


XGBoost version: 3.4.1


In [37]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [38]:
from sklearn.model_selection import train_test_split

X_xgb, _, y_xgb, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.20,
    stratify=y_train,
    random_state=42
)

print("XGBoost training shape:", X_xgb.shape)
print("XGBoost target shape:", y_xgb.shape)
print("\nClass distribution:")
print(y_xgb.value_counts())

XGBoost training shape: (390578, 70)
XGBoost target shape: (390578,)

Class distribution:
Target
0    349560
1     41018
Name: count, dtype: int64


In [39]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

X_train = pd.read_csv(processed_path / "X_train.csv")
y_train = pd.read_csv(processed_path / "y_train.csv").squeeze()

X_test = pd.read_csv(processed_path / "X_test.csv")
y_test = pd.read_csv(processed_path / "y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

X_train: (1952893, 70)
y_train: (1952893,)
X_test : (621371, 70)
y_test : (621371,)


In [40]:
from sklearn.model_selection import train_test_split

X_xgb, _, y_xgb, _ = train_test_split(
    X_train,
    y_train,
    train_size=0.20,
    stratify=y_train,
    random_state=42
)

print("XGBoost training shape:", X_xgb.shape)
print("XGBoost target shape:", y_xgb.shape)

print("\nClass distribution:")
print(y_xgb.value_counts())

XGBoost training shape: (390578, 70)
XGBoost target shape: (390578,)

Class distribution:
Target
0    349560
1     41018
Name: count, dtype: int64


In [41]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

print("XGBoost model created.")

XGBoost model created.


In [42]:
xgb_model.fit(X_xgb, y_xgb)

print("XGBoost training completed.")

XGBoost training completed.


In [43]:
from sklearn.model_selection import train_test_split

X_xgb_train, X_xgb_val, y_xgb_train, y_xgb_val = train_test_split(
    X_xgb,
    y_xgb,
    test_size=0.2,
    stratify=y_xgb,
    random_state=42
)

print("XGBoost training data:", X_xgb_train.shape)
print("XGBoost validation data:", X_xgb_val.shape)
print("Training target:")
print(y_xgb_train.value_counts())
print("\nValidation target:")
print(y_xgb_val.value_counts())

XGBoost training data: (312462, 70)
XGBoost validation data: (78116, 70)
Training target:
Target
0    279648
1     32814
Name: count, dtype: int64

Validation target:
Target
0    69912
1     8204
Name: count, dtype: int64


In [44]:
xgb_model_clean = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model_clean.fit(X_xgb_train, y_xgb_train)

print("Clean XGBoost training completed.")

Clean XGBoost training completed.


In [45]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred_xgb_val = xgb_model_clean.predict(X_xgb_val)

print("XGBoost Validation Results")
print("Accuracy :", accuracy_score(y_xgb_val, y_pred_xgb_val))
print("Precision:", precision_score(y_xgb_val, y_pred_xgb_val))
print("Recall   :", recall_score(y_xgb_val, y_pred_xgb_val))
print("F1 Score :", f1_score(y_xgb_val, y_pred_xgb_val))

print("\nConfusion Matrix:")
print(confusion_matrix(y_xgb_val, y_pred_xgb_val))

XGBoost Validation Results
Accuracy : 0.9996031542833734
Precision: 0.9982928911108402
Recall   : 0.9979278400780107
F1 Score : 0.9981103322157878

Confusion Matrix:
[[69898    14]
 [   17  8187]]


In [46]:
y_prob_xgb_val = xgb_model_clean.predict_proba(X_xgb_val)[:, 1]

print("First 10 XGBoost attack probabilities:")
print(y_prob_xgb_val[:10])

First 10 XGBoost attack probabilities:
[9.9574369e-05 1.3917479e-05 9.4068986e-05 1.0238476e-04 1.2460500e-05
 2.2624834e-05 1.6193959e-05 2.6666217e-05 1.2460500e-05 1.0896588e-04]


In [47]:
y_pred_xgb_test = xgb_model_clean.predict(X_test)

print("XGBoost Friday Test Results")
print("Accuracy :", accuracy_score(y_test, y_pred_xgb_test))
print("Precision:", precision_score(y_test, y_pred_xgb_test))
print("Recall   :", recall_score(y_test, y_pred_xgb_test))
print("F1 Score :", f1_score(y_test, y_pred_xgb_test))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_test))

XGBoost Friday Test Results
Accuracy : 0.671770970965816
Precision: 0.9967543963177151
Recall   : 0.0765032519883327
F1 Score : 0.1420999941110653

Confusion Matrix:
[[400528     55]
 [203897  16891]]


In [48]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    y_pred_threshold = (y_prob_xgb_val >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_xgb_val, y_pred_threshold))
    print("Recall   :", recall_score(y_xgb_val, y_pred_threshold))
    print("F1 Score :", f1_score(y_xgb_val, y_pred_threshold))


Threshold: 0.1
Precision: 0.992614117931953
Recall   : 0.9992686494392979
F1 Score : 0.9959302678734131

Threshold: 0.3
Precision: 0.9970795814066683
Recall   : 0.9987810823988298
F1 Score : 0.9979296066252588

Threshold: 0.5
Precision: 0.9982928911108402
Recall   : 0.9979278400780107
F1 Score : 0.9981103322157878

Threshold: 0.7
Precision: 0.9986545988258317
Recall   : 0.9952462213554364
F1 Score : 0.996947496947497

Threshold: 0.9
Precision: 0.9997533908754624
Recall   : 0.9882983910287665
F1 Score : 0.993992889542724


In [49]:
y_prob_xgb_test = xgb_model_clean.predict_proba(X_test)[:, 1]

print("Friday test probability statistics:")
print("Minimum :", y_prob_xgb_test.min())
print("Maximum :", y_prob_xgb_test.max())
print("Mean    :", y_prob_xgb_test.mean())

print("\nProbability for actual BENIGN traffic:")
print(y_prob_xgb_test[y_test == 0].mean())

print("\nProbability for actual ATTACK traffic:")
print(y_prob_xgb_test[y_test == 1].mean())

Friday test probability statistics:
Minimum : 8.092595e-06
Maximum : 0.99804807
Mean    : 0.055578183

Probability for actual BENIGN traffic:
0.00062156

Probability for actual ATTACK traffic:
0.15528783


In [50]:
xgb_model_weighted = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=8.52,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model_weighted.fit(X_xgb_train, y_xgb_train)

print("Weighted XGBoost training completed.")

Weighted XGBoost training completed.


In [51]:
y_pred_weighted_val = xgb_model_weighted.predict(X_xgb_val)

print("Weighted XGBoost Validation Results")
print("Accuracy :", accuracy_score(y_xgb_val, y_pred_weighted_val))
print("Precision:", precision_score(y_xgb_val, y_pred_weighted_val))
print("Recall   :", recall_score(y_xgb_val, y_pred_weighted_val))
print("F1 Score :", f1_score(y_xgb_val, y_pred_weighted_val))

print("\nConfusion Matrix:")
print(confusion_matrix(y_xgb_val, y_pred_weighted_val))

Weighted XGBoost Validation Results
Accuracy : 0.9993855292129653
Precision: 0.9950230638504491
Recall   : 0.9991467576791809
F1 Score : 0.997080647123221

Confusion Matrix:
[[69871    41]
 [    7  8197]]


In [52]:
y_pred_weighted_test = xgb_model_weighted.predict(X_test)

print("Weighted XGBoost Friday Test Results")
print("Accuracy :", accuracy_score(y_test, y_pred_weighted_test))
print("Precision:", precision_score(y_test, y_pred_weighted_test))
print("Recall   :", recall_score(y_test, y_pred_weighted_test))
print("F1 Score :", f1_score(y_test, y_pred_weighted_test))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_weighted_test))

Weighted XGBoost Friday Test Results
Accuracy : 0.7519147176163676
Precision: 0.9972093300899879
Recall   : 0.30265231806076415
F1 Score : 0.46436898230349866

Confusion Matrix:
[[400396    187]
 [153966  66822]]


In [53]:
y_prob_weighted_test = xgb_model_weighted.predict_proba(X_test)[:, 1]

print("Weighted XGBoost Friday probabilities")
print("Minimum :", y_prob_weighted_test.min())
print("Maximum :", y_prob_weighted_test.max())
print("Mean    :", y_prob_weighted_test.mean())

print("\nActual BENIGN mean probability:")
print(y_prob_weighted_test[y_test == 0].mean())

print("\nActual ATTACK mean probability:")
print(y_prob_weighted_test[y_test == 1].mean())

Weighted XGBoost Friday probabilities
Minimum : 1.6192645e-05
Maximum : 0.99947983
Mean    : 0.081979886

Actual BENIGN mean probability:
0.0017044598

Actual ATTACK mean probability:
0.22762625


In [54]:
y_prob_weighted_val = xgb_model_weighted.predict_proba(X_xgb_val)[:, 1]

print("First 10 weighted XGBoost validation probabilities:")
print(y_prob_weighted_val[:10])

First 10 weighted XGBoost validation probabilities:
[7.3575758e-04 5.7483336e-05 2.0441704e-04 3.5439257e-04 4.9826594e-05
 7.1618844e-05 4.4294964e-05 1.5509383e-04 4.9826594e-05 3.6776095e-04]


In [55]:
for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    y_pred_threshold = (y_prob_weighted_val >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_xgb_val, y_pred_threshold))
    print("Recall   :", recall_score(y_xgb_val, y_pred_threshold))
    print("F1 Score :", f1_score(y_xgb_val, y_pred_threshold))


Threshold: 0.1
Precision: 0.9773566916934812
Recall   : 0.999634324719649
F1 Score : 0.9883699909611329

Threshold: 0.2
Precision: 0.9865238840091445
Recall   : 0.999390541199415
F1 Score : 0.9929155313351499

Threshold: 0.3
Precision: 0.9915346474785343
Recall   : 0.999390541199415
F1 Score : 0.9954470952467674

Threshold: 0.5
Precision: 0.9950230638504491
Recall   : 0.9991467576791809
F1 Score : 0.997080647123221

Threshold: 0.7
Precision: 0.9968377523716857
Recall   : 0.9990248659190639
F1 Score : 0.9979301107999513


In [56]:
y_pred_weighted_test_07 = (y_prob_weighted_test >= 0.7).astype(int)

print("Weighted XGBoost — Friday Test — Threshold 0.7")
print("Accuracy :", accuracy_score(y_test, y_pred_weighted_test_07))
print("Precision:", precision_score(y_test, y_pred_weighted_test_07))
print("Recall   :", recall_score(y_test, y_pred_weighted_test_07))
print("F1 Score :", f1_score(y_test, y_pred_weighted_test_07))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_weighted_test_07))

Weighted XGBoost — Friday Test — Threshold 0.7
Accuracy : 0.6811969016899727
Precision: 0.995069592914176
Recall   : 0.10329365726398174
F1 Score : 0.1871591706434366

Confusion Matrix:
[[400470    113]
 [197982  22806]]


In [57]:
from pathlib import Path

processed_path = Path("../data/processed")

print("Files in processed folder:")
for file in processed_path.iterdir():
    print(file.name)

Files in processed folder:
X_test.csv
X_train.csv
y_test.csv
y_train.csv


In [58]:
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw")

friday_files = [
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

friday_parts = []

for file in friday_files:
    path = raw_path / file
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    friday_parts.append(df)

friday_df = pd.concat(friday_parts, ignore_index=True)

print("Friday dataset shape:", friday_df.shape)
print("\nAttack types:")
print(friday_df["Label"].value_counts())

Friday dataset shape: (703245, 79)

Attack types:
Label
BENIGN      414322
PortScan    158930
DDoS        128027
Bot           1966
Name: count, dtype: int64


In [59]:
print("Raw Friday rows:", len(friday_df))
print("X_test rows:", len(X_test))
print("Predictions:", len(y_pred_weighted_test_07))

print("\nX_test columns:", X_test.shape[1])
print("Friday columns:", friday_df.shape[1])

Raw Friday rows: 703245
X_test rows: 621371
Predictions: 621371

X_test columns: 70
Friday columns: 79


In [60]:
# Make a copy so we don't modify the original Friday dataset
friday_clean = friday_df.copy()

# 1. Clean column names
friday_clean.columns = friday_clean.columns.str.strip()

# 2. Replace infinity values with NaN
import numpy as np

friday_clean = friday_clean.replace([np.inf, -np.inf], np.nan)

# 3. Fill missing values in the two columns we handled earlier
for col in ["Flow Bytes/s", "Flow Packets/s"]:
    if col in friday_clean.columns:
        friday_clean[col] = friday_clean[col].fillna(
            friday_clean[col].median()
        )

# 4. Remove exact duplicate rows
before = len(friday_clean)
friday_clean = friday_clean.drop_duplicates().reset_index(drop=True)
after = len(friday_clean)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", after)
print("Rows removed:", before - after)

Rows before duplicate removal: 703245
Rows after duplicate removal: 615894
Rows removed: 87351


In [61]:
# Columns that were removed during our earlier preprocessing
removed_columns = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

print("Removed columns:")
for col in removed_columns:
    print("-", col)

print("\nFriday clean shape currently:", friday_clean.shape)

print("\nDoes X_test have these columns?")
for col in removed_columns:
    print(col, "->", col in X_test.columns)

Removed columns:
- Bwd PSH Flags
- Bwd URG Flags
- Fwd Avg Bytes/Bulk
- Fwd Avg Packets/Bulk
- Fwd Avg Bulk Rate
- Bwd Avg Bytes/Bulk
- Bwd Avg Packets/Bulk
- Bwd Avg Bulk Rate

Friday clean shape currently: (615894, 79)

Does X_test have these columns?
Bwd PSH Flags -> False
Bwd URG Flags -> False
Fwd Avg Bytes/Bulk -> False
Fwd Avg Packets/Bulk -> False
Fwd Avg Bulk Rate -> False
Bwd Avg Bytes/Bulk -> False
Bwd Avg Packets/Bulk -> False
Bwd Avg Bulk Rate -> False


In [62]:
from pathlib import Path
import pandas as pd

raw_path = Path("../data/raw")

all_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv"
]

parts = []

for file in all_files:
    print("Loading:", file)

    temp = pd.read_csv(raw_path / file)
    temp.columns = temp.columns.str.strip()

    # Store the day/file information
    temp["Day"] = file

    parts.append(temp)

all_df = pd.concat(parts, ignore_index=True)

print("\nCombined raw shape:", all_df.shape)
print("Combined columns:", all_df.shape[1])

Loading: Monday-WorkingHours.pcap_ISCX.csv


Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv

Combined raw shape: (2830743, 80)
Combined columns: 80


In [63]:
# Create binary target
# BENIGN = 0
# Any attack = 1

all_df["Target"] = all_df["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)

# Columns removed during our earlier preprocessing
removed_columns = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Avg Bytes/Bulk",
    "Fwd Avg Packets/Bulk",
    "Fwd Avg Bulk Rate",
    "Bwd Avg Bytes/Bulk",
    "Bwd Avg Packets/Bulk",
    "Bwd Avg Bulk Rate"
]

all_df = all_df.drop(columns=removed_columns)

print("Final cleaned shape:", all_df.shape)
print("\nTarget distribution:")
print(all_df["Target"].value_counts())

Final cleaned shape: (2830743, 73)

Target distribution:
Target
0    2273097
1     557646
Name: count, dtype: int64


In [64]:
print(all_df["Day"].value_counts())


Day
Wednesday-workingHours.pcap_ISCX.csv                           692703
Monday-WorkingHours.pcap_ISCX.csv                              529918
Tuesday-WorkingHours.pcap_ISCX.csv                             445909
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv    288602
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           286467
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv               225745
Friday-WorkingHours-Morning.pcap_ISCX.csv                      191033
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv         170366
Name: count, dtype: int64


In [65]:
# Recreate Logistic Regression predictions at threshold 0.7

threshold = 0.7

y_pred_lr_07 = (y_prob_test >= threshold).astype(int)

print("Logistic Regression predictions recreated.")
print("\nPrediction distribution:")
print(pd.Series(y_pred_lr_07).value_counts())

Logistic Regression predictions recreated.

Prediction distribution:
0    510615
1    110756
Name: count, dtype: int64


In [66]:
# Generate Logistic Regression probabilities on the Friday test set

X_test_scaled_val = scaler_val.transform(X_test)

y_prob_lr_test = model_lr_val.predict_proba(X_test_scaled_val)[:, 1]

# Apply the previously selected validation threshold
threshold = 0.7

y_pred_lr_07 = (y_prob_lr_test >= threshold).astype(int)

print("Logistic Regression predictions recreated.")

print("\nPrediction distribution:")
print(pd.Series(y_pred_lr_07).value_counts())

Logistic Regression predictions recreated.

Prediction distribution:
0    510615
1    110756
Name: count, dtype: int64


In [67]:
print("X_train:", "X_train" in globals())
print("y_train:", "y_train" in globals())
print("X_test:", "X_test" in globals())
print("y_test:", "y_test" in globals())

print("model_lr_val:", "model_lr_val" in globals())
print("scaler_val:", "scaler_val" in globals())
print("xgb_model_weighted:", "xgb_model_weighted" in globals())

X_train: True
y_train: True
X_test: True
y_test: True
model_lr_val: True
scaler_val: True
xgb_model_weighted: True


In [69]:
print("all_df exists:", "all_df" in globals())

if "all_df" in globals():
    print("all_df shape:", all_df.shape)
    print("\nColumns:", all_df.shape[1])
    print("\nTarget distribution:")
    print(all_df["Target"].value_counts())

all_df exists: True
all_df shape: (2830743, 73)

Columns: 73

Target distribution:
Target
0    2273097
1     557646
Name: count, dtype: int64


In [71]:
import gc

if "all_df" in globals():
    del all_df

gc.collect()

print("Large all_df removed from memory.")

Large all_df removed from memory.


In [72]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path("../data/raw")

all_files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv"
]

sample_parts = []

for file in all_files:

    print("Processing:", file)

    df = pd.read_csv(raw_path / file)

    # Clean column names and labels
    df.columns = df.columns.str.strip()
    df["Label"] = df["Label"].str.strip()

    # Replace infinity only in the two problematic columns
    for col in ["Flow Bytes/s", "Flow Packets/s"]:
        if col in df.columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(df[col].median())

    # Remove exact duplicates within this file
    df = df.drop_duplicates()

    # Create binary target
    df["Target"] = (df["Label"] != "BENIGN").astype("int8")

    # Remove constant columns
    removed_columns = [
        "Bwd PSH Flags",
        "Bwd URG Flags",
        "Fwd Avg Bytes/Bulk",
        "Fwd Avg Packets/Bulk",
        "Fwd Avg Bulk Rate",
        "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk",
        "Bwd Avg Bulk Rate"
    ]

    df = df.drop(columns=removed_columns)

    # Take at most 10% from each file
    sample_size = max(1, int(len(df) * 0.10))

    sample = df.sample(
        n=sample_size,
        random_state=42
    )

    sample_parts.append(sample)

    print("  Cleaned rows:", len(df))
    print("  Sampled rows:", len(sample))

    # Free memory before next file
    del df
    gc.collect()

# Combine only the samples
experiment_b_df = pd.concat(
    sample_parts,
    ignore_index=True
)

print("\nExperiment B sample created.")
print("Shape:", experiment_b_df.shape)

print("\nTarget distribution:")
print(experiment_b_df["Target"].value_counts())

Processing: Monday-WorkingHours.pcap_ISCX.csv
  Cleaned rows: 502983
  Sampled rows: 50298
Processing: Tuesday-WorkingHours.pcap_ISCX.csv
  Cleaned rows: 421844
  Sampled rows: 42184
Processing: Wednesday-workingHours.pcap_ISCX.csv
  Cleaned rows: 610794
  Sampled rows: 61079
Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Cleaned rows: 252972
  Sampled rows: 25297
Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Cleaned rows: 164300
  Sampled rows: 16430
Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Cleaned rows: 223112
  Sampled rows: 22311
Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Cleaned rows: 214114
  Sampled rows: 21411
Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Cleaned rows: 184145
  Sampled rows: 18414

Experiment B sample created.
Shape: (257424, 72)

Target distribution:
Target
0    214587
1     42837
Name: count, dtype: int64


In [73]:
feature_columns = [
    col for col in experiment_b_df.columns
    if col not in ["Label", "Target"]
]

X_expB = experiment_b_df[feature_columns]
y_expB = experiment_b_df["Target"]

print("X_expB shape:", X_expB.shape)
print("y_expB shape:", y_expB.shape)

print("\nTarget distribution:")
print(y_expB.value_counts())

X_expB shape: (257424, 70)
y_expB shape: (257424,)

Target distribution:
Target
0    214587
1     42837
Name: count, dtype: int64


In [74]:
from sklearn.model_selection import train_test_split

X_expB_train, X_expB_val, y_expB_train, y_expB_val = train_test_split(
    X_expB,
    y_expB,
    test_size=0.20,
    stratify=y_expB,
    random_state=42
)

print("Training shape:", X_expB_train.shape)
print("Validation shape:", X_expB_val.shape)

print("\nTraining target distribution:")
print(y_expB_train.value_counts())

print("\nValidation target distribution:")
print(y_expB_val.value_counts())

Training shape: (205939, 70)
Validation shape: (51485, 70)

Training target distribution:
Target
0    171669
1     34270
Name: count, dtype: int64

Validation target distribution:
Target
0    42918
1     8567
Name: count, dtype: int64


In [75]:
from sklearn.preprocessing import StandardScaler

scaler_expB = StandardScaler()

X_expB_train_scaled = scaler_expB.fit_transform(X_expB_train)
X_expB_val_scaled = scaler_expB.transform(X_expB_val)

print("Scaled training shape:", X_expB_train_scaled.shape)
print("Scaled validation shape:", X_expB_val_scaled.shape)

Scaled training shape: (205939, 70)
Scaled validation shape: (51485, 70)


In [76]:
from sklearn.linear_model import LogisticRegression

model_lr_expB = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    n_jobs=-1
)

model_lr_expB.fit(
    X_expB_train_scaled,
    y_expB_train
)

print("Experiment B Logistic Regression training completed.")

d:\secureAccessControl\.venv-1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Experiment B Logistic Regression training completed.


In [77]:
y_pred_expB = model_lr_expB.predict(X_expB_val_scaled)

print("Prediction completed.")
print("\nPrediction distribution:")
print(pd.Series(y_pred_expB).value_counts())

Prediction completed.

Prediction distribution:
0    40034
1    11451
Name: count, dtype: int64


In [78]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy_expB = accuracy_score(y_expB_val, y_pred_expB)
precision_expB = precision_score(y_expB_val, y_pred_expB)
recall_expB = recall_score(y_expB_val, y_pred_expB)
f1_expB = f1_score(y_expB_val, y_pred_expB)

cm_expB = confusion_matrix(y_expB_val, y_pred_expB)

print("Experiment B - Logistic Regression")
print("-----------------------------------")
print(f"Accuracy : {accuracy_expB:.4f} ({accuracy_expB*100:.2f}%)")
print(f"Precision: {precision_expB:.4f} ({precision_expB*100:.2f}%)")
print(f"Recall   : {recall_expB:.4f} ({recall_expB*100:.2f}%)")
print(f"F1-score : {f1_expB:.4f} ({f1_expB*100:.2f}%)")

print("\nConfusion Matrix:")
print(cm_expB)

print("\nClassification Report:")
print(classification_report(
    y_expB_val,
    y_pred_expB,
    target_names=["BENIGN", "ATTACK"]
))

Experiment B - Logistic Regression
-----------------------------------
Accuracy : 0.9361 (93.61%)
Precision: 0.7305 (73.05%)
Recall   : 0.9764 (97.64%)
F1-score : 0.8357 (83.57%)

Confusion Matrix:
[[39832  3086]
 [  202  8365]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       0.99      0.93      0.96     42918
      ATTACK       0.73      0.98      0.84      8567

    accuracy                           0.94     51485
   macro avg       0.86      0.95      0.90     51485
weighted avg       0.95      0.94      0.94     51485



In [79]:
print("========== Logistic Regression Comparison ==========")

print("\nExperiment A - Friday Unseen-Day / Unseen-Attack Test")
print(f"Accuracy : 81.85%")
print(f"Precision: 98.77%")
print(f"Recall   : 49.54%")
print(f"F1-score : 65.99%")

print("\nExperiment B - Random Stratified Validation")
print(f"Accuracy : {accuracy_expB*100:.2f}%")
print(f"Precision: {precision_expB*100:.2f}%")
print(f"Recall   : {recall_expB*100:.2f}%")
print(f"F1-score : {f1_expB*100:.2f}%")

========== Logistic Regression Comparison ==========

Experiment A - Friday Unseen-Day / Unseen-Attack Test
Accuracy : 81.85%
Precision: 98.77%
Recall   : 49.54%
F1-score : 65.99%

Experiment B - Random Stratified Validation
Accuracy : 93.61%
Precision: 73.05%
Recall   : 97.64%
F1-score : 83.57%


In [80]:
from sklearn.ensemble import RandomForestClassifier

model_rf_expB = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model_rf_expB.fit(
    X_expB_train,
    y_expB_train
)

print("Experiment B Random Forest training completed.")

Experiment B Random Forest training completed.


In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_pred_rf_expB = model_rf_expB.predict(X_expB_val)

accuracy_rf_expB = accuracy_score(y_expB_val, y_pred_rf_expB)
precision_rf_expB = precision_score(y_expB_val, y_pred_rf_expB)
recall_rf_expB = recall_score(y_expB_val, y_pred_rf_expB)
f1_rf_expB = f1_score(y_expB_val, y_pred_rf_expB)

cm_rf_expB = confusion_matrix(y_expB_val, y_pred_rf_expB)

print("Experiment B - Random Forest")
print("--------------------------------")
print(f"Accuracy : {accuracy_rf_expB:.4f} ({accuracy_rf_expB*100:.2f}%)")
print(f"Precision: {precision_rf_expB:.4f} ({precision_rf_expB*100:.2f}%)")
print(f"Recall   : {recall_rf_expB:.4f} ({recall_rf_expB*100:.2f}%)")
print(f"F1-score : {f1_rf_expB:.4f} ({f1_rf_expB*100:.2f}%)")

print("\nConfusion Matrix:")
print(cm_rf_expB)

Experiment B - Random Forest
--------------------------------
Accuracy : 0.9983 (99.83%)
Precision: 0.9941 (99.41%)
Recall   : 0.9957 (99.57%)
F1-score : 0.9949 (99.49%)

Confusion Matrix:
[[42867    51]
 [   37  8530]]


In [82]:
from xgboost import XGBClassifier

model_xgb_expB = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model_xgb_expB.fit(
    X_expB_train,
    y_expB_train
)

print("Experiment B XGBoost training completed.")

Experiment B XGBoost training completed.


In [83]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_pred_xgb_expB = model_xgb_expB.predict(X_expB_val)

accuracy_xgb_expB = accuracy_score(y_expB_val, y_pred_xgb_expB)
precision_xgb_expB = precision_score(y_expB_val, y_pred_xgb_expB)
recall_xgb_expB = recall_score(y_expB_val, y_pred_xgb_expB)
f1_xgb_expB = f1_score(y_expB_val, y_pred_xgb_expB)

cm_xgb_expB = confusion_matrix(y_expB_val, y_pred_xgb_expB)

print("Experiment B - XGBoost")
print("-----------------------")
print(f"Accuracy : {accuracy_xgb_expB:.4f} ({accuracy_xgb_expB*100:.2f}%)")
print(f"Precision: {precision_xgb_expB:.4f} ({precision_xgb_expB*100:.2f}%)")
print(f"Recall   : {recall_xgb_expB:.4f} ({recall_xgb_expB*100:.2f}%)")
print(f"F1-score : {f1_xgb_expB:.4f} ({f1_xgb_expB*100:.2f}%)")

print("\nConfusion Matrix:")
print(cm_xgb_expB)

Experiment B - XGBoost
-----------------------
Accuracy : 0.9988 (99.88%)
Precision: 0.9960 (99.60%)
Recall   : 0.9966 (99.66%)
F1-score : 0.9963 (99.63%)

Confusion Matrix:
[[42884    34]
 [   29  8538]]


In [84]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    
    "Experiment A Accuracy (%)": [
        81.85,
        64.74,
        68.12
    ],
    
    "Experiment A Precision (%)": [
        98.77,
        94.42,
        99.51
    ],
    
    "Experiment A Recall (%)": [
        49.54,
        0.83,
        10.33
    ],
    
    "Experiment A F1 (%)": [
        65.99,
        1.64,
        18.72
    ],
    
    "Experiment B Accuracy (%)": [
        accuracy_expB * 100,
        accuracy_rf_expB * 100,
        accuracy_xgb_expB * 100
    ],
    
    "Experiment B Precision (%)": [
        precision_expB * 100,
        precision_rf_expB * 100,
        precision_xgb_expB * 100
    ],
    
    "Experiment B Recall (%)": [
        recall_expB * 100,
        recall_rf_expB * 100,
        recall_xgb_expB * 100
    ],
    
    "Experiment B F1 (%)": [
        f1_expB * 100,
        f1_rf_expB * 100,
        f1_xgb_expB * 100
    ]
})

print(comparison.round(2).to_string(index=False))

              Model  Experiment A Accuracy (%)  Experiment A Precision (%)  Experiment A Recall (%)  Experiment A F1 (%)  Experiment B Accuracy (%)  Experiment B Precision (%)  Experiment B Recall (%)  Experiment B F1 (%)
Logistic Regression                      81.85                       98.77                    49.54                65.99                      93.61                       73.05                    97.64                83.57
      Random Forest                      64.74                       94.42                     0.83                 1.64                      99.83                       99.41                    99.57                99.49
            XGBoost                      68.12                       99.51                    10.33                18.72                      99.88                       99.60                    99.66                99.63


In [85]:
y_prob_xgb_expB = model_xgb_expB.predict_proba(X_expB_val)[:, 1]

print("Probability calculation completed.")

print("\nProbability statistics:")
print(f"Minimum : {y_prob_xgb_expB.min():.6f}")
print(f"Maximum : {y_prob_xgb_expB.max():.6f}")
print(f"Mean    : {y_prob_xgb_expB.mean():.6f}")

Probability calculation completed.

Probability statistics:
Minimum : 0.000017
Maximum : 0.999930
Mean    : 0.166378


In [86]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

for threshold in thresholds:

    y_pred_threshold = (y_prob_xgb_expB >= threshold).astype(int)

    precision = precision_score(y_expB_val, y_pred_threshold)
    recall = recall_score(y_expB_val, y_pred_threshold)
    f1 = f1_score(y_expB_val, y_pred_threshold)

    print(f"\nThreshold: {threshold}")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall   : {recall * 100:.2f}%")
    print(f"F1 Score : {f1 * 100:.2f}%")


Threshold: 0.1
Precision: 98.85%
Recall   : 99.92%
F1 Score : 99.38%

Threshold: 0.3
Precision: 99.33%
Recall   : 99.85%
F1 Score : 99.59%

Threshold: 0.5
Precision: 99.60%
Recall   : 99.66%
F1 Score : 99.63%

Threshold: 0.7
Precision: 99.68%
Recall   : 99.30%
F1 Score : 99.49%

Threshold: 0.9
Precision: 99.69%
Recall   : 98.65%
F1 Score : 99.17%


In [87]:
from sklearn.metrics import confusion_matrix, classification_report

threshold_expB = 0.5

y_pred_xgb_expB_threshold = (
    y_prob_xgb_expB >= threshold_expB
).astype(int)

cm_expB = confusion_matrix(
    y_expB_val,
    y_pred_xgb_expB_threshold
)

print("Threshold:", threshold_expB)

print("\nConfusion Matrix:")
print(cm_expB)

print("\nClassification Report:")
print(
    classification_report(
        y_expB_val,
        y_pred_xgb_expB_threshold,
        target_names=["BENIGN", "ATTACK"]
    )
)

Threshold: 0.5

Confusion Matrix:
[[42884    34]
 [   29  8538]]

Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     42918
      ATTACK       1.00      1.00      1.00      8567

    accuracy                           1.00     51485
   macro avg       1.00      1.00      1.00     51485
weighted avg       1.00      1.00      1.00     51485



In [88]:
import pandas as pd

results_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Experiment": "A - Unseen Friday",
        "Accuracy": 81.85,
        "Precision": 98.77,
        "Recall": 49.54,
        "F1": 65.99
    },
    {
        "Model": "Random Forest",
        "Experiment": "A - Unseen Friday",
        "Accuracy": 64.74,
        "Precision": 94.42,
        "Recall": 0.83,
        "F1": 1.64
    },
    {
        "Model": "XGBoost",
        "Experiment": "A - Unseen Friday",
        "Accuracy": 68.12,
        "Precision": 99.51,
        "Recall": 10.33,
        "F1": 18.72
    },
    {
        "Model": "Logistic Regression",
        "Experiment": "B - Random Validation",
        "Accuracy": 93.61,
        "Precision": 73.05,
        "Recall": 97.64,
        "F1": 83.57
    },
    {
        "Model": "Random Forest",
        "Experiment": "B - Random Validation",
        "Accuracy": 99.83,
        "Precision": 99.41,
        "Recall": 99.57,
        "F1": 99.49
    },
    {
        "Model": "XGBoost",
        "Experiment": "B - Random Validation",
        "Accuracy": 99.88,
        "Precision": 99.60,
        "Recall": 99.66,
        "F1": 99.63
    }
])

print(results_summary.to_string(index=False))

              Model            Experiment  Accuracy  Precision  Recall    F1
Logistic Regression     A - Unseen Friday     81.85      98.77   49.54 65.99
      Random Forest     A - Unseen Friday     64.74      94.42    0.83  1.64
            XGBoost     A - Unseen Friday     68.12      99.51   10.33 18.72
Logistic Regression B - Random Validation     93.61      73.05   97.64 83.57
      Random Forest B - Random Validation     99.83      99.41   99.57 99.49
            XGBoost B - Random Validation     99.88      99.60   99.66 99.63


In [93]:
from pathlib import Path

print("Current folder:")
print(Path.cwd())

print("\nBackend folder exists?")
print(Path("../backend").exists())

print("\nModels folder exists?")
print(Path("../backend/models").exists())

Current folder:
d:\secureAccessControl\notebooks

Backend folder exists?
True

Models folder exists?
False


In [94]:
model_xgb_expB.save_model("../backend/models/xgboost_model.json")

In [95]:
import json

with open("../backend/models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=4)

print("Feature columns saved successfully.")
print("Number of features:", len(feature_columns))

Feature columns saved successfully.
Number of features: 70
